# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.### Dataset SourceThe dataset source is provided via a Croissant schema URL:`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data LoadingLoad metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pdimport jsonfrom pprint import pprint# Define the dataset URLcroissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'# Load the dataset metadatadataset = mlc.Dataset(croissant_url)metadata = dataset.metadataprint(f"Name: {metadata.name}")print(f"Description: {metadata.description}")

## 2. Data OverviewReview available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their @idprint('Available Record Sets:')record_sets = []for record_set in metadata.record_sets:    print(f" - {record_set['@id']}")    record_sets.append(record_set['@id'])# For each record set, print fields and columns (by @id also)for record_set in metadata.record_sets:    print(f"\nRecord Set '@id': {record_set['@id']}")    print(f"  Name: {record_set.get('name', '[No name]')}")    if 'field' in record_set:        print('  Fields:')        for field in record_set['field']:            print(f"    - @id: {field['@id']}, name: {field.get('name','[No name]')}, dataType: {field.get('dataType','')}")            # If this field is extracted from a column, print column @id            if 'column' in field:                if isinstance(field['column'], list):                    for column in field['column']:                        print(f"        column @id: {column['@id']}")                else:                    print(f"        column @id: {field['column']['@id']}")

## 3. Data ExtractionLoad data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Assuming the main record set contains the protein quantitative results.# For this dataset, we check the metadata to find appropriate record set IDs.# If you're unsure, run the cell above for record set and field @ids and fill the variables below.# Example for this dataset:# Let's suppose the main record set @id is 'cr:proteinQuantitativeResults', override as found via metadata (see previous cell)!if len(record_sets) == 0:    # Fallback: try to discover them programmatically    record_sets = [rs['@id'] for rs in metadata.record_sets]# Print to let users check and changeprint('Record set IDs:', record_sets)dataframes = {}for record_set_id in record_sets:    print(f"\n--- Reading record set: {record_set_id} ---")    try:        records = list(dataset.records(record_set=record_set_id))        if records:            # Try to load to a dataframe (if records are dict-like, likely they will be based on @id fields)            df = pd.DataFrame(records)            dataframes[record_set_id] = df            print(f"Loaded DataFrame shape: {df.shape}")            print(f"Columns: {df.columns.tolist()}")            print(df.head(3))        else:            print(f"No records found for {record_set_id}")    except Exception as e:        print(f"Could not load records for {record_set_id}: {e}")# Pick a record set for the subsequent analysis (change as appropriate):main_record_set_id = Nonefor r_id in dataframes:    if dataframes[r_id].shape[1] > 2:        main_record_set_id = r_id        print(f"Selected {r_id} for detailed analysis.")        breakif main_record_set_id is None and dataframes:    main_record_set_id = list(dataframes)[0]    print(f"Defaulting to {main_record_set_id}")print(f"\nUsing record set: {main_record_set_id}")if main_record_set_id:    print('Columns:', dataframes[main_record_set_id].columns.tolist())    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations such as removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# EDA for protein abundance field (you may adjust field names according to overview cell above).# Example fields in this dataset (look for numeric fields like 'Abundance', 'coverage', 'MW (kDa)', etc.).# Here, replace 'cr:abundance' and 'cr:gene' with actual @id from column list.df = dataframes[main_record_set_id]# Try to auto-detect a numeric field for illustration:numeric_candidates = [col for col in df.columns if df[col].dtype.kind in 'iuf']if not numeric_candidates:    # Try object columns that might be numeric    for col in df.columns:        try:            df[col+'_parsed'] = pd.to_numeric(df[col], errors='coerce')            if df[col+'_parsed'].notnull().sum() > 0:                numeric_candidates.append(col)        except Exception:            continueif len(numeric_candidates) == 0:    print("No numeric field detected in the current DataFrame. Please update the field name below.")    numeric_field = df.columns[0]else:    numeric_field = numeric_candidates[0]print(f"Using numeric field: {numeric_field}")# Filter based on chosen numeric field (e.g., only proteins with abundance > threshold)threshold = df[numeric_field].quantile(0.75) if df[numeric_field].dtype.kind in 'iuf' else 10filtered_df = df[df[numeric_field] > threshold]print(f"Filtered records with {numeric_field} > {threshold}:")print(filtered_df.head())# Normalize the chosen numeric column:filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()print(f"Normalized {numeric_field} for filtered records:")print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())# Try to find a grouping field, e.g., 'cr:gene' or any categorical field.group_candidates = [col for col in df.columns if df[col].dtype == 'object']if group_candidates:    group_field = group_candidates[0]    print(f"Grouping by field: {group_field}")    # Take mean of normalized field by group    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)    print(grouped_df.head())else:    group_field = None

## 5. VisualizationVisualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as pltimport seaborn as sns# Histogram of numeric fieldplt.figure(figsize=(8, 4))sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)plt.title(f"Distribution of {numeric_field}")plt.xlabel(numeric_field)plt.show()# Boxplot grouped by group_field if available.if group_field:    high_cardinality = df[group_field].nunique() > 20    if not high_cardinality:        plt.figure(figsize=(12, 5))        sns.boxplot(x=df[group_field], y=df[numeric_field])        plt.title(f"{numeric_field} by {group_field}")        plt.xticks(rotation=45)        plt.show()

## 6. ConclusionSummarize key findings and observations from the dataset exploration.- The dataset enabled programmatic access and exploration of protein abundance and related molecular data.- We identified numeric and categorical fields by their `@id` values, and used filtering and normalization for basic EDA.- Example visualizations highlighted the distribution of protein properties and, where possible, group-wise differences.*You can continue your analysis by customizing numeric and category fields based on the IDs reviewed in section 2.*